# TealKit MCP Agent — Weather Forecast MCP — Qwen2.5-1.5B Native Ollama Bootstrap Notebook

This notebook is the runnable bootstrap path for the `ollama_native` contract for the Weather Forecast MCP server.

**Tool Set:** `get_current_weather`, `get_hourly_forecast`, `get_daily_forecast`, `geocode_weather_city`

**Key Features:**
1. ChatML format with Qwen2.5-1.5B (32K native context).
2. Configurable context window (8K/16K/32K/64K).
3. Native Ollama tool_calls format (no legacy text wrappers).

## Cell 1 - Install Dependencies

In [ ]:
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers" trl peft accelerate bitsandbytes datasets huggingface_hub

import shutil
shutil.rmtree('/root/.unsloth', ignore_errors=True)

print('Install done. Restarting runtime...')
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

## Cell 2 - Native Contract Config
> Set model, paths, and context window. Adjust `MAX_SEQ_LENGTH` based on GPU memory.

In [ ]:
from pathlib import Path

SERVER_SCOPE = 'weatherforecast'
CONTRACT_TYPE = 'ollama_native'
MODEL_PRESET = 'qwen2_5_1_5b'
MODEL_NAME = 'unsloth/Qwen2.5-1.5B-Instruct'
MODEL_SLUG = 'qwen25-1p5b-weatherforecast-ollama'
HF_REPO = f'lschaffer/{MODEL_SLUG}'  # change if needed
PROMPT_CONTRACT = 'contracts/weatherforecast/ollama_native/prompt_contract.md'
QUALITY_GATE_PROFILE = 'weatherforecast_ollama_native'

# ── Context Window ──────────────────────────────────────────────────────────────
# Supported values for Qwen2.5-1.5B-Instruct:
#   8192   (8K)    — fits any GPU (T4/L4/H100)
#   16384  (16K)   — fits L4/H100, default
#   32768  (32K)   — native max, needs H100 or reduced batch size
#   65536  (64K)   — requires RoPE extension (YaRN), H100 recommended
MAX_SEQ_LENGTH = 16384

# ── Drive paths ───────────────────────────────────────────────────────────────
drive_root = Path('/content/drive/MyDrive/Tealkit/training') / SERVER_SCOPE
DATA_DIR = drive_root / 'datasets' / CONTRACT_TYPE
TRAIN_FILE = DATA_DIR / 'train_split.jsonl'
VALID_FILE = DATA_DIR / 'valid_split.jsonl'

OUTPUT_DIR = drive_root / 'mcp_adapters_qwen25_1p5b_ollama'
MERGE_DIR  = drive_root / 'mcp_merged_model_qwen25_1p5b_ollama'
GGUF_DIR   = drive_root / 'mcp_fused_model_qwen25_1p5b_ollama'

DRIVE_SYSTEM_PROMPT_FILE = DATA_DIR / 'ollama_native_system_prompt.md'

DEFAULT_NATIVE_SYSTEM_PROMPT = (
    'You are a Weather Assistant with access to global weather data. '
    'Use tools accurately, keep tool-call turns concise, and keep final '
    'answers grounded in returned tool results.'
)
SYSTEM_PROMPT = DEFAULT_NATIVE_SYSTEM_PROMPT

print('SERVER_SCOPE :', SERVER_SCOPE)
print('CONTRACT_TYPE:', CONTRACT_TYPE)
print('MODEL_NAME   :', MODEL_NAME)
print('MAX_SEQ_LENGTH:', MAX_SEQ_LENGTH)
print('Train file   :', TRAIN_FILE)
print('Valid file   :', VALID_FILE)
print('HF repo      :', HF_REPO)


## Cell 3 - Mount Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

if DRIVE_SYSTEM_PROMPT_FILE.is_file():
    SYSTEM_PROMPT = DRIVE_SYSTEM_PROMPT_FILE.read_text(encoding='utf-8').strip()
    print('System prompt loaded from Drive:', DRIVE_SYSTEM_PROMPT_FILE)
else:
    SYSTEM_PROMPT = DEFAULT_NATIVE_SYSTEM_PROMPT
    print('System prompt: using default (no override on Drive)')

for check_path, label, required in [
    (TRAIN_FILE,               'train_split.jsonl',              True),
    (VALID_FILE,               'valid_split.jsonl',              True),
    (DRIVE_SYSTEM_PROMPT_FILE, 'drive native system prompt override', False),
]:
    if Path(check_path).is_file():
        print('OK      ', label, check_path)
    elif required:
        print('MISSING ', label, check_path)
    else:
        print('OPTIONAL', label, check_path)


## Cell 4 - Inspect Native Contract Inputs

In [ ]:
import json

print('Train file :', TRAIN_FILE)
print('Valid file :', VALID_FILE)
print('Output dir :', OUTPUT_DIR)
print('GGUF dir   :', GGUF_DIR)
print()
print('System prompt:')
print(SYSTEM_PROMPT[:500])
print()

if TRAIN_FILE.is_file():
    with open(TRAIN_FILE, encoding='utf-8') as f:
        first = json.loads(f.readline())
    print('First train example keys:', list(first.keys()))
else:
    print('Train file not yet present — mount Drive and upload the splits first.')


## Cell 5 - Load Base Model

In [ ]:
from unsloth import FastLanguageModel
import torch
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    dtype=torch.bfloat16,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template='chatml',
)

print('Base model loaded.')

## Cell 6 - Apply LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
model.print_trainable_parameters()

## Cell 7 - Load Dataset

In [ ]:
import json
from datasets import Dataset, DatasetDict

def load_clean_jsonl(filepath):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            # Normalize mixed types for tool calls to prevent PyArrow parse errors
            for msg in row.get('messages', []):
                if 'tool_calls' in msg:
                    for tc in msg['tool_calls']:
                        func = tc.get('function', {})
                        args = func.get('arguments', {})
                        if args:
                            for k in ['latitude', 'longitude']:
                                if k in args and args[k] is not None:
                                    try:
                                        args[k] = float(args[k])
                                    except (ValueError, TypeError):
                                        pass
            data.append(row)
    return Dataset.from_list(data)

dataset = DatasetDict({
    'train': load_clean_jsonl(TRAIN_FILE),
    'validation': load_clean_jsonl(VALID_FILE)
})


def ensure_system_message(messages):
    if messages and isinstance(messages[0], dict) and messages[0].get('role') == 'system':
        return messages
    return [{'role': 'system', 'content': SYSTEM_PROMPT}] + list(messages)

def format_example(examples):
    texts = []
    for messages in examples['messages']:
        normalized = ensure_system_message(messages)
        texts.append(
            tokenizer.apply_chat_template(
                normalized, tokenize=False, add_generation_prompt=False
            )
        )
    return {'text': texts}

dataset = dataset.map(format_example, batched=True)

def detect_chat_markers(sample_text):
    instruction_candidates = [
        '<|im_start|>user\\n',
        '<|im_start|>user<|im_sep|>',
        '<|user|>',
    ]
    response_candidates = [
        '<|im_start|>assistant\\n',
        '<|im_start|>assistant<|im_sep|>',
        '<|assistant|>',
    ]
    instruction_part = next((part for part in instruction_candidates if part in sample_text), None)
    response_part = next((part for part in response_candidates if part in sample_text), None)
    return instruction_part, response_part

sample_text = dataset['train'][0]['text'] if len(dataset['train']) else ''
INSTRUCTION_PART, RESPONSE_PART = detect_chat_markers(sample_text)

print('Train examples:', len(dataset['train']))
print('Valid examples:', len(dataset['validation']))
print('Instruction marker:', repr(INSTRUCTION_PART))
print('Response marker   :', repr(RESPONSE_PART))
print(sample_text[:2000])

## Cell 8 - Training
> Adjust batch_size based on MAX_SEQ_LENGTH (auto-calculated below).

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq
from unsloth.chat_templates import train_on_responses_only

_batch_size = 4 if MAX_SEQ_LENGTH <= 8192 else (2 if MAX_SEQ_LENGTH <= 16384 else 1)
_grad_accum = 2 if MAX_SEQ_LENGTH <= 8192 else (4 if MAX_SEQ_LENGTH <= 32768 else 8)

trainer_args = SFTConfig(
    per_device_train_batch_size=_batch_size,
    gradient_accumulation_steps=_grad_accum,
    warmup_steps=5,
    max_steps=250,
    learning_rate=5e-5,
    logging_steps=5,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='no',
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    seed=3407,
    output_dir='/content/outputs',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    packing=False,
    args=trainer_args,
)

if INSTRUCTION_PART and RESPONSE_PART:
    try:
        masked_trainer = train_on_responses_only(
            trainer,
            instruction_part=INSTRUCTION_PART,
            response_part=RESPONSE_PART,
        )
        if len(masked_trainer.train_dataset) == 0:
            print('WARNING: Response masking removed every train sample. Falling back to full-sequence training.')
        else:
            trainer = masked_trainer
            print(f'Response masking active: {len(trainer.train_dataset)} train samples.')
    except Exception as exc:
        print('WARNING: Response masking failed, using full sequence. Error:', exc)
else:
    print('WARNING: Could not detect chat markers in formatted text. Using full-sequence training.')

trainer.train()
print('Training complete.')

## Cell 9 - Export
> Saves adapters, tries native GGUF export, falls back to llama.cpp conversion.

In [ ]:
import gc
import glob
import os
import shutil
import subprocess
import sys

OUTPUT_DIR   = str(OUTPUT_DIR)
MERGE_DIR    = str(MERGE_DIR)
GGUF_DIR     = str(GGUF_DIR)

QUANT_METHOD = 'q4_k_m'
GGUF_BASENAME = f"{HF_REPO.split('/')[-1]}-unsloth"
GGUF_F16_PATH = os.path.join(GGUF_DIR, f'{GGUF_BASENAME}-F16.gguf')
GGUF_QUANT_PATH = os.path.join(GGUF_DIR, f'{GGUF_BASENAME}-{QUANT_METHOD.upper()}.gguf')
FINAL_GGUF_FILE = None
GGUF_FILENAME = None

os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Adapters successfully saved to Drive:', OUTPUT_DIR)

def run_checked(command, cwd=None, extra_env=None):
    env = os.environ.copy()
    for key in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP', 'PYTHONUSERBASE'):
        env.pop(key, None)
    env['PYTHONNOUSERSITE'] = '1'
    if extra_env:
        for key, value in extra_env.items():
            if value is None:
                env.pop(key, None)
            else:
                env[key] = value
    command = [str(c) for c in command]
    print('>>', ' '.join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

def clear_unsloth_llama_cpp_cache():
    cache_dir = '/root/.unsloth/llama.cpp'
    if os.path.isdir(cache_dir):
        shutil.rmtree(cache_dir, ignore_errors=True)
        print('Cleared cached Unsloth llama.cpp checkout:', cache_dir)

def refresh_tokenizer_files(merged_dir, base_model):
    from huggingface_hub import hf_hub_download
    for filename in ('tokenizer_config.json', 'tokenizer.json', 'special_tokens_map.json'):
        try:
            source_path = hf_hub_download(repo_id=base_model, filename=filename)
            shutil.copy2(source_path, os.path.join(merged_dir, filename))
        except Exception as exc:
            print(f'INFO: Could not refresh {filename}: {exc}')
    try:
        source_path = hf_hub_download(repo_id=base_model, filename='tokenizer.model')
        shutil.copy2(source_path, os.path.join(merged_dir, 'tokenizer.model'))
    except Exception:
        pass

def find_quantize_binary(llama_cpp_dir):
    candidates = [
        os.path.join(llama_cpp_dir, 'build', 'bin', 'llama-quantize'),
        os.path.join(llama_cpp_dir, 'build', 'bin', 'quantize'),
        shutil.which('llama-quantize'),
        shutil.which('quantize'),
    ]
    for candidate in candidates:
        if candidate and os.path.isfile(candidate) and os.access(candidate, os.X_OK):
            return candidate
    return None

def ensure_quantize_binary(llama_cpp_dir):
    quantize_bin = find_quantize_binary(llama_cpp_dir)
    if quantize_bin:
        return quantize_bin
    run_checked(['cmake', '-S', '.', '-B', 'build', '-DBUILD_SHARED_LIBS=OFF', '-DGGML_CUDA=OFF'], cwd=llama_cpp_dir)
    run_checked(['cmake', '--build', 'build', '--config', 'Release', '-j2'], cwd=llama_cpp_dir)
    return find_quantize_binary(llama_cpp_dir)

def install_llama_cpp_requirements(llama_cpp_dir, pydeps_dir):
    requirements_file = os.path.join(llama_cpp_dir, 'requirements.txt')
    if os.path.isdir(pydeps_dir):
        shutil.rmtree(pydeps_dir)
    os.makedirs(pydeps_dir, exist_ok=True)
    if os.path.isfile(requirements_file):
        run_checked([sys.executable, '-m', 'pip', 'install', '--upgrade', '--target', pydeps_dir, '-r', requirements_file])

def ensure_llama_cpp_checkout(llama_cpp_dir):
    convert_script = os.path.join(llama_cpp_dir, 'convert_hf_to_gguf.py')
    if not os.path.isdir(os.path.join(llama_cpp_dir, '.git')):
        if os.path.exists(llama_cpp_dir):
            shutil.rmtree(llama_cpp_dir, ignore_errors=True)
        run_checked(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp', llama_cpp_dir])
    return convert_script

def manual_llama_cpp_convert(merged_dir):
    clear_unsloth_llama_cpp_cache()
    refresh_tokenizer_files(merged_dir, MODEL_NAME)
    llama_cpp_dir = '/content/llama.cpp'
    pydeps_dir = '/content/llama_cpp_pydeps'
    convert_script = ensure_llama_cpp_checkout(llama_cpp_dir)
    gguf_py_dir = os.path.join(llama_cpp_dir, 'gguf-py')
    install_llama_cpp_requirements(llama_cpp_dir, pydeps_dir)
    isolated_pythonpath = os.pathsep.join([p for p in [pydeps_dir, gguf_py_dir, llama_cpp_dir] if p])
    quantize_bin = ensure_quantize_binary(llama_cpp_dir)
    run_checked([sys.executable, '-S', convert_script, merged_dir, '--outfile', GGUF_F16_PATH, '--outtype', 'f16'], cwd=llama_cpp_dir, extra_env={'PYTHONPATH': isolated_pythonpath})
    if quantize_bin:
        run_checked([quantize_bin, GGUF_F16_PATH, GGUF_QUANT_PATH, QUANT_METHOD.upper()])
        return GGUF_QUANT_PATH
    return GGUF_F16_PATH

if os.path.exists(GGUF_DIR):
    shutil.rmtree(GGUF_DIR)
os.makedirs(GGUF_DIR, exist_ok=True)

try:
    clear_unsloth_llama_cpp_cache()
    trainer.model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method=QUANT_METHOD)
    gguf_files = sorted(glob.glob(f'{GGUF_DIR}/*.gguf'))
    if not gguf_files:
        raise RuntimeError(f'No GGUF files were created in {GGUF_DIR}')
    FINAL_GGUF_FILE = gguf_files[0]
    GGUF_FILENAME = os.path.basename(FINAL_GGUF_FILE)
    print('Native GGUF export complete:', FINAL_GGUF_FILE)
except Exception as exc:
    print('Native GGUF export failed, falling back to manual llama.cpp conversion:', exc)
    if os.path.exists(MERGE_DIR):
        shutil.rmtree(MERGE_DIR)
    trainer.model.save_pretrained_merged(MERGE_DIR, tokenizer, save_method='merged_16bit')
    FINAL_GGUF_FILE = manual_llama_cpp_convert(MERGE_DIR)
    GGUF_FILENAME = os.path.basename(FINAL_GGUF_FILE)

if not FINAL_GGUF_FILE or not os.path.isfile(FINAL_GGUF_FILE):
    raise RuntimeError('GGUF export failed: no .gguf file was created.')

print('Final GGUF file:', FINAL_GGUF_FILE)
torch.cuda.empty_cache()
gc.collect()

## Cell 10 - Native Evaluation
> Validates with the Ollama-native quality gate profile.

In [ ]:
MODEL_CARD_PATH = f'{GGUF_DIR}/README.md'
GGUF_FILENAME = globals().get('GGUF_FILENAME') or f"{HF_REPO.split('/')[-1]}-unsloth-{QUANT_METHOD.upper()}.gguf"

model_card_content = f'''---
base_model: {MODEL_NAME}
library_name: unsloth
tags:
- mcp
- weather-forecast
- tool-calling
- ollama-native
- gguf
---

# Weather Forecast MCP Agent - {MODEL_NAME.split('/')[-1]}

This model was fine-tuned for the native Ollama contract of Weather Forecast MCP.

## Files
- GGUF: {GGUF_FILENAME}
- Adapters: saved during notebook execution

## Contract
- Prompt contract: {PROMPT_CONTRACT}
- Quality gate profile: {QUALITY_GATE_PROFILE}
'''

with open(MODEL_CARD_PATH, 'w', encoding='utf-8') as handle:
    handle.write(model_card_content)

print('Model card generated at:', MODEL_CARD_PATH)

# ── Generate Modelfile ──
MODELFILE_PATH = f'{GGUF_DIR}/Modelfile'
modelfile_content = f'''FROM {GGUF_FILENAME}

TEMPLATE """{{{{ if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ end }}}}{{{{ if .Prompt }}}}<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
<|im_start|>assistant
{{{{ end }}}}"""

PARAMETER stop "<|im_end|>"
PARAMETER stop "<|endoftext|>"
PARAMETER num_ctx {MAX_SEQ_LENGTH}
PARAMETER num_thread 4
'''

with open(MODELFILE_PATH, 'w', encoding='utf-8') as handle:
    handle.write(modelfile_content)
print('Modelfile generated at:', MODELFILE_PATH)
print('Quality gate profile:', QUALITY_GATE_PROFILE)

UPLOAD_TO_HF = False  # set to True when you want to upload immediately

if UPLOAD_TO_HF:
    import getpass
    from huggingface_hub import HfApi, upload_folder
    try:
        hf_token = os.environ.get('HF_TOKEN') or getpass.getpass('HF token: ')
        api = HfApi(token=hf_token)
        api.create_repo(repo_id=HF_REPO, repo_type='model', exist_ok=True)
        upload_folder(
            repo_id=HF_REPO,
            folder_path=GGUF_DIR,
            repo_type='model',
            token=hf_token,
        )
        print('Uploaded GGUF folder to Hugging Face:', HF_REPO)
    except Exception as exc:
        print('HF upload skipped/failed:', exc)